# dataset: loading square crops (M3)

Loads the square crops written by `capture.ipynb` (format in `docs/contracts.md`):
`<crops_root>/<session_id>/manifest.csv` plus `squares/*.png`.
**Splits are always by session_id, never by frame**, so the test set really contains
unseen lighting / camera placements.

torch is imported lazily, so everything except `build_transforms` and the torch Dataset
works without it.

Milestone: M3 (square classifiers).

**TODO** (see `docs/M3_TODO.md`)
- [ ] `build_transforms`: settle the augmentation set (flips, perspective, blur).
- [ ] `load_chessred`: ChessReD loader (optional pretraining data).
- [ ] `load_chesscog_synthetic`: chesscog synthetic loader (optional pretraining data).

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

%run ./labels.ipynb

In [ ]:
MANIFEST_COLUMNS = ("session_id", "game_id", "move_idx", "frame_id", "square", "label", "path", "fen")

## Manifests
`load_manifest` concatenates the sessions' manifests and adds the training targets:
`y` (13-class index), `y_occ` (0/1) and `y_color3` (0 empty / 1 white / 2 black), plus
`abs_path` (crop path resolved against the session directory).

In [ ]:
def list_sessions(crops_root: str | Path) -> list[str]:
    """Session ids under crops_root (directories that contain a manifest.csv)."""
    root = Path(crops_root)
    return sorted(p.parent.name for p in root.glob("*/manifest.csv"))


def load_manifest(crops_root: str | Path, sessions: Iterable[str] | None = None) -> pd.DataFrame:
    """Concatenate manifest.csv of the given sessions (all sessions if None).

    Adds columns: `abs_path` (crop path resolved against the session dir), `y` (13-class
    index), `y_occ` (0/1), `y_color3` (0 empty / 1 white / 2 black).
    """
    root = Path(crops_root)
    sessions = list_sessions(root) if sessions is None else list(sessions)
    if not sessions:
        raise ValueError(f"no sessions selected under {root}")

    frames = []
    for s in sessions:
        mpath = root / s / "manifest.csv"
        if not mpath.exists():
            raise FileNotFoundError(mpath)
        df = pd.read_csv(mpath, dtype={"frame_id": str, "game_id": str, "session_id": str})
        missing = set(MANIFEST_COLUMNS) - set(df.columns)
        if missing:
            raise ValueError(f"{mpath} missing columns {sorted(missing)}")
        df["abs_path"] = [str(root / s / p) for p in df["path"]]
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)

    bad = sorted(set(df["label"]) - set(CLASSES))
    if bad:
        raise ValueError(f"unknown labels in manifest: {bad}")
    df["y"] = df["label"].map(CLASS_TO_INDEX).astype(int)
    df["y_occ"] = df["label"].map(label_to_occupancy).astype(int)
    df["y_color3"] = df["label"].map(label_to_color3).astype(int)
    return df


def split_by_session(cfg) -> dict[str, pd.DataFrame]:
    """Return {"train", "val", "test"} DataFrames from cfg.data.*_sessions.

    Empty session lists give empty DataFrames. Raises if a session is in two splits.
    """
    d = cfg.data
    splits = {"train": d.train_sessions or [], "val": d.val_sessions or [], "test": d.test_sessions or []}
    names = list(splits)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            overlap = set(splits[a]) & set(splits[b])
            if overlap:
                raise ValueError(f"sessions {sorted(overlap)} appear in both {a} and {b}")
    return {
        k: load_manifest(d.crops_root, v) if v else pd.DataFrame(columns=list(MANIFEST_COLUMNS))
        for k, v in splits.items()
    }


def read_crop(path: str | Path) -> np.ndarray:
    """Read a crop as an RGB uint8 array (H, W, 3)."""
    import cv2

    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## torch side
Transforms come from `cfg.cnn.input_size` and `cfg.cnn.augment`. Eval: resize -> tensor ->
ImageNet normalize; train adds augmentation. `SquareCropDataset` falls back to a plain
`object` base class when torch is missing so this notebook still runs.

In [ ]:
def build_transforms(cfg, train: bool):
    """torchvision transform for crops, from cfg.cnn.input_size and cfg.cnn.augment.

    Eval: resize -> tensor -> ImageNet normalize. Train adds augmentation.
    """
    from torchvision import transforms as T

    h, w = cfg.cnn.input_size
    norm = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    steps = [T.ToPILImage(), T.Resize((h, w))]
    if train:
        aug = cfg.cnn.augment
        # TODO M3: decide on the final augmentation set. Horizontal flips are label-safe for a
        # single piece; consider also RandomPerspective / blur to mimic session differences.
        steps += [
            T.ColorJitter(brightness=aug.color_jitter, contrast=aug.color_jitter,
                          saturation=aug.color_jitter),
            T.RandomAffine(degrees=aug.rotation_deg, translate=(aug.translate, aug.translate)),
        ]
    steps += [T.ToTensor(), norm]
    return T.Compose(steps)


def _torch_dataset_base():
    try:
        from torch.utils.data import Dataset
    except ImportError:  # notebook must run without torch
        return object
    return Dataset


class SquareCropDataset(_torch_dataset_base()):
    """torch Dataset over a manifest DataFrame. Item: (image tensor, class index).

    `target` picks the label column: "y" (13 classes), "y_occ" or "y_color3".
    """

    def __init__(self, df: pd.DataFrame, transform=None, target: str = "y"):
        self.paths = df["abs_path"].tolist()
        self.targets = df[target].astype(int).tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int):
        img = read_crop(self.paths[i])
        if self.transform is not None:
            img = self.transform(img)
        return img, self.targets[i]


def class_weights(df: pd.DataFrame) -> np.ndarray:
    """Inverse-frequency weights (13,) for the loss; empty squares dominate real data.

    Classes absent from df get weight 0.
    """
    counts = np.bincount(df["y"].to_numpy(dtype=int), minlength=len(CLASSES)).astype(float)
    w = np.zeros_like(counts)
    nz = counts > 0
    w[nz] = counts.sum() / (nz.sum() * counts[nz])
    return w

## External datasets (optional pretraining)

In [ ]:
def load_chessred(root: str | Path, split: str = "train") -> pd.DataFrame:
    """ChessReD (Masouris & van Gemert 2023): real photos of full boards.

    Source format: `images/<game>/*.jpg` and one `annotations.json` (COCO-like) with per-image
    piece annotations (category + chessboard_position like "e4") and, for a subset, the four
    board corners. Splits are given in the JSON.

    TODO M3: for images with corner annotations, warp with homography.ipynb (warp_board) and
    cut crops with the same geometry as capture.ipynb (extract_all_crops), then return a
    DataFrame with the manifest columns (session_id = "chessred_<game>") so it plugs into
    SquareCropDataset. Map ChessReD category names to CLASSES. Probably cache crops to disk
    once rather than cropping on the fly.
    """
    raise NotImplementedError("TODO M3: ChessReD loader")


def load_chesscog_synthetic(root: str | Path, split: str = "train") -> pd.DataFrame:
    """chesscog synthetic dataset (Wölflein & Arandjelović 2021): Blender renders.

    Source format: `<split>/NNNN.png` with a matching `NNNN.json` holding `fen`, `corners`
    (4 board corners in pixels), `white_turn` and lighting metadata.

    TODO M3: warp each render with the corners, crop the 64 squares like capture.ipynb, derive
    labels from the FEN via board_to_labels, and return a manifest-style DataFrame
    (session_id = "synth"). Used when cfg.cnn.pretrain_on == "synthetic".
    """
    raise NotImplementedError("TODO M3: chesscog synthetic loader")

## Self-check
(was tests/test_dataset.py). Fake sessions are written to a temporary directory.

TODO notes from skipped tests:
- `build_transforms(cfg, train=True/False)` on a (200, 100, 3) uint8 crop gives a float tensor
  (3, H, W) with H, W = cfg.cnn.input_size; the train transform is random, eval is not.
- `load_chessred` on a tiny fixture returns manifest columns with labels in CLASSES.
- `load_chesscog_synthetic`: 64 rows per render, labels match board_to_labels(FEN).

In [ ]:
import tempfile
from types import SimpleNamespace


def _ds_write_session(root, session, n_frames=1, board=None):
    """Fake session: manifest only (64 rows per frame), crops are not written."""
    board = board or chess.Board()
    labels = board_to_labels(board)
    sdir = Path(root) / session
    sdir.mkdir(parents=True)
    lines = ["session_id,game_id,move_idx,frame_id,square,label,path,fen"]
    for f in range(n_frames):
        fid = f"{f:06d}"
        for sq in range(64):
            name = chess.SQUARE_NAMES[sq]
            lines.append(f"{session},g1,{f},{fid},{name},{labels[sq]},squares/{fid}_{name}.png,{board.fen()}")
    (sdir / "manifest.csv").write_text("\n".join(lines) + "\n")


def _ds_cfg(root, train, val=(), test=()):
    return SimpleNamespace(data=SimpleNamespace(
        crops_root=str(root), train_sessions=list(train), val_sessions=list(val),
        test_sessions=list(test)))


def _ds_raises(exc, fn, *a, match=None):
    try:
        fn(*a)
    except exc as e:
        assert match is None or match in str(e), e
        return
    raise AssertionError(f"{fn.__name__} did not raise {exc.__name__}")


# load_manifest labels
with tempfile.TemporaryDirectory() as _t:
    _t = Path(_t)
    _ds_write_session(_t, "s01", n_frames=2)
    _df = load_manifest(_t, ["s01"])
    assert len(_df) == 128
    assert _df["frame_id"].iloc[0] == "000000"  # kept as str
    _e1 = _df[(_df.square == "e1") & (_df.frame_id == "000000")].iloc[0]
    assert _e1.label == "K" and _e1.y == CLASS_TO_INDEX["K"] and _e1.y_occ == 1 and _e1.y_color3 == 1
    assert _df[_df.square == "e8"].iloc[0].y_color3 == 2
    assert (_df[_df.label == "empty"].y == 0).all()
    assert _e1.abs_path.endswith("s01/squares/000000_e1.png")

    # class weights: 32 empties vs 1 king
    _w = class_weights(load_manifest(_t, ["s01"]))
    assert _w.shape == (13,) and _w[CLASS_TO_INDEX["empty"]] < _w[CLASS_TO_INDEX["K"]]

# list sessions and load all
with tempfile.TemporaryDirectory() as _t:
    _t = Path(_t)
    _ds_write_session(_t, "s02")
    _ds_write_session(_t, "s01")
    assert list_sessions(_t) == ["s01", "s02"]
    assert set(load_manifest(_t)["session_id"]) == {"s01", "s02"}

# unknown label rejected, missing session
with tempfile.TemporaryDirectory() as _t:
    _t = Path(_t)
    _ds_write_session(_t, "s01")
    _m = _t / "s01" / "manifest.csv"
    _m.write_text(_m.read_text().replace(",K,", ",X,"))
    _ds_raises(ValueError, load_manifest, _t, ["s01"], match="unknown labels")
    _ds_raises(FileNotFoundError, load_manifest, _t, ["nope"])

# split by session, overlap rejected
with tempfile.TemporaryDirectory() as _t:
    _t = Path(_t)
    for _s in ("a", "b", "c"):
        _ds_write_session(_t, _s)
    _splits = split_by_session(_ds_cfg(_t, ["a", "b"], ["c"]))
    assert set(_splits["train"].session_id) == {"a", "b"}
    assert set(_splits["val"].session_id) == {"c"}
    assert len(_splits["test"]) == 0
    _ds_raises(ValueError, split_by_session, _ds_cfg(_t, ["a"], ["a"]), match="both")

# SquareCropDataset needs torch (and cv2)
try:
    import torch  # noqa: F401
    import cv2
    with tempfile.TemporaryDirectory() as _t:
        _t = Path(_t)
        _ds_write_session(_t, "s01")
        _df = load_manifest(_t, ["s01"]).head(3)
        (_t / "s01" / "squares").mkdir(exist_ok=True)
        for _p in _df.abs_path:
            cv2.imwrite(_p, np.zeros((200, 100, 3), np.uint8))
        _dset = SquareCropDataset(_df)
        _img, _y = _dset[0]
        assert len(_dset) == 3 and _img.shape == (200, 100, 3) and _y == _df.y.iloc[0]
except ImportError:
    print("skipped SquareCropDataset check (torch not installed)")
print("dataset OK")